# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Deep Learning Para Aplicações de IA com PyTorch e Lightning</font>

## <font color='blue'>Mini-Projeto 1</font>
## <font color='blue'>Aplicação de IA Para Análise de Sentimento de Textos em Português</font>

![DSA](images/MP1.png)

## Instalando e Carregando os Pacotes

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
#!pip install -q -U watermark

In [ ]:
#!pip install -q torch==2.0.0
!pip install -q torch

In [ ]:
#!pip install -q transformers==4.28.1
!pip install -q transformers

In [ ]:
# Imports
import csv
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

In [ ]:
from transformers import logging
logging.set_verbosity_error()

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

## Carregando os Dados

In [ ]:
# Caminho do arquivo CSV que você deseja ler
csv_file_path = 'data/frases.csv'

In [ ]:
# Crie uma lista vazia para armazenar as frases
frases = []

In [ ]:
# Abra o arquivo CSV no modo leitura ('r') e use o objeto 'csv.reader' para ler o arquivo
with open(csv_file_path, 'r', encoding='utf-8') as file:
    
    csv_reader = csv.reader(file)

    # Iterar sobre cada linha no arquivo CSV
    for row in csv_reader:
        frase = row[0]
        frases.append(frase)

In [ ]:
# Imprime a lista de frases
print(frases)

In [ ]:
# Nome do objeto 
texts = frases

In [ ]:
# 1: positivo, 0: negativo
labels = [1, 0, 1, 0, 1, 1, 0, 1, 1, 0]  

## Pré-Processamento

In [ ]:
RANDOM_SEED = 42

In [ ]:
# Divisão dos dados em treino e teste
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, 
                                                                      labels, 
                                                                      test_size = 0.2, 
                                                                      random_state = RANDOM_SEED)

In [ ]:
# Carrega o tokenizador
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# Classe de tokenização dos dados
class SentimentAnalysisTokenizer(Dataset):
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        
        text = self.texts[idx]
        label = self.labels[idx]
        inputs = self.tokenizer.encode_plus(text,
                                            add_special_tokens = True,
                                            max_length = self.max_length,
                                            padding = 'max_length',
                                            truncation = True,
                                            return_tensors = 'pt')

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(label)
        }

In [ ]:
MAX_LENGTH = 64

In [ ]:
# Aplica a tokenização
train_dataset = SentimentAnalysisTokenizer(train_texts, train_labels, tokenizer, MAX_LENGTH)

In [ ]:
# Aplica a tokenização
test_dataset = SentimentAnalysisTokenizer(test_texts, test_labels, tokenizer, MAX_LENGTH)

In [ ]:
# Cria o data loader
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)

In [ ]:
# Cria o data loader
test_loader = DataLoader(test_dataset, batch_size = 16)

## Loop de Treino, Avaliação e Inferência

In [ ]:
# Função para treinar o modelo
def train_epoch(model, data_loader, criterion, optimizer, device):
    
    model.train()
    total_loss = 0

    for batch in data_loader:
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask = attention_mask, labels = labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)

In [ ]:
# Função para avaliar o modelo
def eval_epoch(model, data_loader, criterion, device):
    
    model.eval()
    total_loss = 0

    with torch.no_grad():
        
        for batch in data_loader:
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

    return total_loss / len(data_loader)

In [ ]:
# Função para obter previsões do modelo
def predict(model, data_loader, device):
    
    model.eval()
    predictions = []

    with torch.no_grad():
        
        for batch in data_loader:
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.tolist())

    return predictions

## Construção do Modelo

In [ ]:
# Inicialização do modelo e configuração do dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Carrega o modelo
modelo = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels = 2)

In [ ]:
# Coloca o modelo na memória do device
modelo.to(device)

In [ ]:
# Hiperparâmetros
EPOCHS = 10
LEARNING_RATE = 2e-5

In [ ]:
# Otimizador
optimizer = torch.optim.AdamW(modelo.parameters(), lr = LEARNING_RATE)

**torch.optim.AdamW()** é uma classe de otimizador no PyTorch que implementa a versão AdamW do algoritmo de otimização Adam. AdamW é uma extensão do otimizador Adam que inclui correções de decaimento de peso (Weight Decay) para melhorar a generalização do modelo em tarefas de aprendizado profundo.

Aqui está uma descrição dos principais parâmetros da classe torch.optim.AdamW:

- params (iterável): Um iterável de tensores (geralmente os parâmetros do modelo) que serão otimizados.


- lr (float, opcional): Taxa de aprendizado (learning rate), um hiperparâmetro que controla o tamanho do passo de atualização dos parâmetros do modelo. O valor padrão é 1e-3 (0.001).


- betas (par de floats, opcional): Coeficientes usados para calcular as médias móveis exponenciais dos gradientes e seus quadrados. Os valores padrão são (0.9, 0.999).


- eps (float, opcional): Termo adicionado para melhorar a estabilidade numérica durante a otimização. O valor padrão é 1e-8.


-  weight_decay (float, opcional): Coeficiente de decaimento de peso (Weight Decay), um hiperparâmetro que penaliza os parâmetros do modelo para evitar overfitting. O valor padrão é 0.


- amsgrad (bool, opcional): Se True, usa a variante AMSGrad do otimizador Adam, que é mais robusta em certos casos, mas geralmente não é necessário. O valor padrão é False.

In [ ]:
# Função de perda
criterion = torch.nn.CrossEntropyLoss()

**torch.nn.CrossEntropyLoss()** é uma classe de função de perda (loss function) no PyTorch, usada para tarefas de classificação multi-classe. A função de perda de entropia cruzada combina a função LogSoftmax e a função de perda NLLLoss (Negative Log-Likelihood Loss) em uma única classe, tornando-a conveniente e eficiente para uso em problemas de classificação.

A entropia cruzada mede a diferença entre duas distribuições de probabilidade, neste caso, a distribuição prevista pelo modelo e a distribuição real (rótulos verdadeiros). Durante o treinamento, o objetivo é minimizar a perda de entropia cruzada, o que leva a um melhor ajuste do modelo aos dados.

## Treinamento e Avaliação do Modelo

In [ ]:
# Treinamento e validação do modelo
for epoch in range(EPOCHS):
    train_loss = train_epoch(modelo, train_loader, criterion, optimizer, device)
    test_loss = eval_epoch(modelo, test_loader, criterion, device)
    print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss}, Test Loss: {test_loss}')

In [ ]:
# Salva o modelo em disco
torch.save(modelo, 'models/modelo_dsa_mp1.pt')

In [ ]:
# Carrega o modelo do disco
modelo_final = torch.load('models/modelo_dsa_mp1.pt')

## Testando o Modelo com Novos Dados

In [ ]:
# Novos dados
novas_frases = ['Eu gostei muito deste filme.',
                'O atendimento do restaurante foi decepcionante.']

In [ ]:
# Aplica a tokenização
dataset = SentimentAnalysisTokenizer(novas_frases, [0] * len(novas_frases), tokenizer, MAX_LENGTH)

In [ ]:
# Cria o data loader
loader = DataLoader(dataset, batch_size = 16)

In [ ]:
# Previsões
previsoes = predict(modelo_final, loader, device)

In [ ]:
# Análise de sentimento
for text, prediction in zip(novas_frases, previsoes):
    print(f'Sentença: {text} | Sentimento: {"positivo" if prediction else "negativo"}')

# Fim